# Dimer Case II: Functions

This page contains parameter values and functions used in other pages of this chapter.

**Import packages**

In [1]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
from qutip import *

**Create a data class**

In [2]:
from dataclasses import dataclass
@dataclass
class Results:
    times: object
    populations: object
    states: object
    expect: object

**Wavelength from frequency**

In [3]:
def wavelength(omega0):
    k0 = omega0  # omega = c k, c=1
    lambda0 = 2*np.pi/k0
    return k0, lambda0

**Spin operators in 2^2 = 4 dimensions**

In [4]:
# Spin operators ---
def spin_ops():
    i2 = qeye(2)
    sz = [tensor(sigmaz(),i2),tensor(i2,sigmaz())]
    sp = [tensor(sigmap(),i2),tensor(i2,sigmap())]
    sm = [tensor(sigmam(),i2),tensor(i2,sigmam())]
    return sz, sp, sm

**Product basis**

In [5]:
def pbasis():
    #--This function calls kprod3--
    # v = product basis vectors [8][8]
    # label = names of corresponding states [8]
    label=["ee","eg","ge","gg"]
    v=[]
    v.append(tensor(basis(2,0),basis(2,0)))
    v.append(tensor(basis(2,0),basis(2,1)))
    v.append(tensor(basis(2,1),basis(2,0)))
    v.append(tensor(basis(2,1),basis(2,1)))

    return v, label

**Dicke basis**

In [6]:
def dicke(v):
    #--- v = product basis [4][4]
    #--- u = Dicke basis [4][4]
    label=["e","s","a","g"]
    u=[]
    u.append(v[0])
    u.append(1/np.sqrt(2)*(v[1]+v[2]))
    u.append(1/np.sqrt(2)*(v[1]-v[2]))
    u.append(v[3]) 
    return u, label

**Dipole-dipole coupling**


In [7]:
def ddcoupling(x,theta):
    y=(1-np.cos(theta)**2)*np.cos(x)/x + (1-3*np.sin(theta)**2)*(np.sin(x)/x**2 - np.cos(x)/x**3)
    return -y*3/4

**Hamiltonian**

In [8]:
def hamiltonian():
    return 0*tensor(qeye(2),qeye(2))

**Decay rates**

In [9]:
def decay_rate(x,chi):
    y=(1-np.cos(chi)**2)*np.sin(x)/x + (1-3*np.cos(chi)**2)*(np.cos(x)/x**2 - np.sin(x)/x**3)
    gamma = [(1+y),(1-y)]
    return gamma

**Collapse operators**

In [10]:
def collapse_ops(P,gamma,gamma_phi,u):

    sz, sp, sm = spin_ops()
    
    c_ops = []
    
#-- pumping
    c_ops.append(np.sqrt(P)*sp[0])
    c_ops.append(np.sqrt(P)*sp[1])

    #-- emission    
    [e,s,a,g]=[0,1,2,3]
    # superradient channel
    L = u[s]*u[e].dag()+u[g]*u[s].dag()
    c_ops.append(np.sqrt(gamma[0])*L)
    # subradient channel
    L = u[a]*u[e].dag()+u[g]*u[a].dag()
    c_ops.append(np.sqrt(gamma[1])*L)

#-- pure dephasing
    c_ops.append(np.sqrt(gamma_phi)*sz[0])
    c_ops.append(np.sqrt(gamma_phi)*sz[1])
    
    return c_ops

**Position of emitters**

In [11]:
def emitter_pos(a):
    r1 = [a/2,0,0]
    r2 = [-a/2,0,0]
    return [r1,r2]

**Location of detector**

In [12]:
def detector_pos(theta,phi):
    return np.array([np.sin(theta)*np.cos(phi),np.sin(theta)*np.sin(phi),np.cos(theta)])

**Rate matrix**

In [13]:
def rate_matrix(P, gamma):
    return np.array([
        [-2*gamma,       P,       P,       0],
        [ gamma, -(gamma+P),      0,       P],
        [ gamma,       0, -(gamma+P),      P],
        [     0,   gamma,   gamma,    -2*P]
    ], dtype=float)

In [14]:
def steadypops(P, gamma):
    D = (P+gamma)**2
    Q1 = P**2/D
    Q2 = P*gamma/D
    Q3 = Q2
    Q4 = gamma**2/D
    Q =np.array([Q1,Q2,Q3,Q4])
    v, label = pbasis()
    PI = []
    for k in range(4):
        PI.append(ket2dm(v[k]))

    rho = sum(Q*PI)
    
    return Q, rho

**Population dynamics solver**

In [15]:
def popsolve(M,pop0,times,e_ops=[]):

    n_times = times.size
    n_ops = len(e_ops)
   
    Mt = times[:, np.newaxis, np.newaxis] * M
    expMt = linalg.expm(Mt)
    pop = expMt @ pop0

    #--- converting population to QuTiP density operator
    v, label = pbasis()
    P = []
    for k in range(4):
        P.append(ket2dm(v[k]))
    rho = []
    for k in range(n_times):
        Q = pop[k]
        QP = sum(Q*P)
        rho.append(QP)

#--- evaluating expectation value

        
    exval = []
    for n in range(n_ops):
        val = []
        for k in range(n_times):
            val.append(expect(e_ops[0],rho[k]))
        exval.append(val)
        
    return Results(times,pop,rho,exval)
